<a href="https://colab.research.google.com/github/anitsirh-C/DRA-workshhop/blob/main/IgFold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **IgFold**: Fast, accurate antibody structure prediction

Official notebook for [IgFold](https://www.biorxiv.org/content/10.1101/2022.04.20.488972): Fast, accurate antibody structure prediction from deep learning on massive set of natural antibodies.  The code, data, and weights for this work are made available for non-commercial use. For commercial inquiries, please contact `jruffolo[at]jhu.edu`.

In [1]:
#@title Input antibody Fv sequences then press `Runtime` -> `Run all`

import os
import sys

python_version = "3.10"

name = "my_antibody" #@param {type:"string"}
pred_dir = name
os.makedirs(pred_dir, exist_ok=True)

#@markdown Enter antibody sequences for structure prediction. To predict a nanobody structure (or an individual heavy or light chain), simply provide one sequence.
heavy_sequence = "EVQLVQSGPEVKKPGTSVKVSCKASGFTFMSSAVQWVRQARGQRLEWIGWIVIGSGNTNYAQKFQERVTITRDMSTSTAYMELSSLRSEDTAVYYCAAPYCSSISCNDGFDIWGQGTMVTVS" #@param {type:"string"}
light_sequence = "DVVMTQTPFSLPVSLGDQASISCRSSQSLVHSNGNTYLHWYLQKPGQSPKLLIYKVSNRFSGVPDRFSGSGSGTDFTLKISRVEAEDLGVYFCSQSTHVPYTFGGGTKLEIK" #@param {type:"string"}

sequences = {}
if len(heavy_sequence) > 0:
    sequences["H"] = heavy_sequence
if len(light_sequence) > 0:
    sequences["L"] = light_sequence

#@markdown Perform structural refinement with OpenMM
do_refine = True #@param {type:"boolean"}
#@markdown Renumber predicted antibody structure (Chothia) with AbNumber
do_renum = True #@param {type:"boolean"}
#@markdown Use only a single model for predictions (instead of model ensemble)
single_model = False #@param {type:"boolean"}

In [2]:
import os

# Remove all '.READY' files to force a complete re-installation
for filename in ['CONDA_READY', 'CODE_READY', 'AMBER_READY', 'ABNUMBER_READY']:
    if os.path.isfile(filename):
        os.remove(filename)
        print(f"Removed {filename} to force re-installation.")
    else:
        print(f"{filename} not found, no need to remove.")

print("Please run the 'Install dependencies' cell (cell id: LsJNdVE87Go2) again to re-install all packages.")

CONDA_READY not found, no need to remove.
CODE_READY not found, no need to remove.
AMBER_READY not found, no need to remove.
ABNUMBER_READY not found, no need to remove.
Please run the 'Install dependencies' cell (cell id: LsJNdVE87Go2) again to re-install all packages.


In [3]:
#@title Install dependencies hallo

# Use system python version (currently 3.12 in Colab)
# The python_version variable is defined in the first input cell

if not os.path.isfile("CONDA_READY"):
  print("installing conda...")
  conda_script_name = "Mambaforge-24.9.2-0-Linux-x86_64.sh"
  # Ensure previous download attempts are cleaned up
  os.system(f"rm -f {conda_script_name}")
  # Try to download and log any wget errors - explicitly output to the desired filename
  wget_cmd = f"wget -O {conda_script_name} https://github.com/conda-forge/miniforge/releases/download/24.9.2-0/{conda_script_name} > conda_install_log.txt 2>&1"
  os.system(wget_cmd)

  # Check if the download was successful
  if not os.path.isfile(conda_script_name):
      # If the file wasn't downloaded, log an explicit error
      with open("conda_install_log.txt", "a") as f:
          f.write(f"ERROR: Failed to download {conda_script_name}. Please check the URL and your internet connection.\n")
      print(f"ERROR: Mambaforge installer not downloaded. See conda_install_log.txt for details.")
  else:
      # If downloaded, proceed with installation
      os.system(f"bash {conda_script_name} -bfp /usr/local >> conda_install_log.txt 2>&1")
      # Add mamba to PATH before trying to use it
      os.environ["PATH"] = "/usr/local/bin:" + os.environ["PATH"]
      os.system("mamba config --set auto_update_conda false >> conda_install_log.txt 2>&1") # Log this too
      os.system("touch CONDA_READY")
      print("Conda installation process completed. See conda_install_log.txt for full details.")
  os.system("cat conda_install_log.txt") # Always print the log at the end for immediate feedback.

if not os.path.isfile("CODE_READY"):
  print("installing igfold...")
  # Force uninstall existing torch/torchvision to ensure clean slate for new version
  os.system(f"/usr/bin/python3 -m pip uninstall -y torch torchvision > igfold_install_log.txt 2>&1")

  # Install torch and torchvision (from specified index) first, explicitly requiring >= 2.6
  # Use /usr/bin/python3 to target Colab's default Python 3.12, and use its globally defined python_version for consistency
  os.system(f"/usr/bin/python3 -m pip install --verbose --force-reinstall 'torch>=2.6.0' torchvision --index-url https://download.pytorch.org/whl/cu121 >> igfold_install_log.txt 2>&1")

  # Then install igfold (from PyPI)
  os.system(f"/usr/bin/python3 -m pip install --verbose 'igfold>=0.3.0' >> igfold_install_log.txt 2>&1")

  # Install other dependencies, appending their output to the log
  os.system(f"/usr/bin/python3 -m pip install -q --no-warn-conflicts 'py3Dmol>=2.0.1' matplotlib seaborn >> igfold_install_log.txt 2>&1")

  # Finally, print the combined log and create the ready file
  os.system("cat igfold_install_log.txt")
  os.system("touch CODE_READY")

if do_refine and not os.path.isfile("AMBER_READY"):
  print("installing amber...")
  # Removed openmm version constraint to allow mamba to find compatible version with active Python
  os.system(f"mamba install -y -q -c conda-forge openmm python='{python_version}' pdbfixer > amber_install_log.txt 2>&1") # Use active python_version
  os.system("cat amber_install_log.txt") # Print the log file
  os.system("touch AMBER_READY")

if do_renum and not os.path.isfile("ABNUMBER_READY"):
  print("installing abnumber...")
  # Added -y flag for non-interactive installation and redirect output to abnumber_install_log.txt
  os.system(f"mamba install -y -q -c bioconda abnumber python='{python_version}' > abnumber_install_log.txt 2>&1") # Use active python_version
  os.system("pip install pandas --force-reinstall >> abnumber_install_log.txt 2>&1") # Also capture pandas install
  os.system("cat abnumber_install_log.txt") # Print the log file
  os.system("touch ABNUMBER_READY")

installing conda...
Conda installation process completed. See conda_install_log.txt for full details.
installing igfold...
installing amber...
installing abnumber...


In [4]:
import os

if os.path.exists('conda_install_log.txt'):
    with open('conda_install_log.txt', 'r') as f:
        print(f.read())
else:
    print("Error: conda_install_log.txt not found. Please ensure cell LsJNdVE87Go2 was executed successfully after the last modification.")

--2026-01-08 01:50:02--  https://github.com/conda-forge/miniforge/releases/download/24.9.2-0/Mambaforge-24.9.2-0-Linux-x86_64.sh
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/221584272/804fab16-af8c-4c38-a3ed-51e5712ed646?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-01-08T02%3A33%3A11Z&rscd=attachment%3B+filename%3DMambaforge-24.9.2-0-Linux-x86_64.sh&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-01-08T01%3A32%3A12Z&ske=2026-01-08T02%3A33%3A11Z&sks=b&skv=2018-11-09&sig=nabZUNys3626CCkT5jsWMBajIkTxyOVQGZmChcdL380%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc2NzgzODgwMiwibmJmIjoxNzY3ODM3MDAyLC

In [5]:
import os

conda_script_name = "Mambaforge-24.9.2-0-Linux-x86_64.sh"
if os.path.isfile(conda_script_name):
    print(f"Die Datei '{conda_script_name}' wurde gefunden. Größe: {os.path.getsize(conda_script_name)} Bytes.")
else:
    print(f"Fehler: Die Datei '{conda_script_name}' wurde nicht gefunden. Der Download ist fehlgeschlagen oder unvollständig.")

Die Datei 'Mambaforge-24.9.2-0-Linux-x86_64.sh' wurde gefunden. Größe: 78136824 Bytes.


In [9]:
print(wget_cmd)
print("Bitte überprüfe diese URL in deinem Browser, um sicherzustellen, dass sie noch gültig ist und die Datei heruntergeladen werden kann.")

wget -O Mambaforge-24.9.2-0-Linux-x86_64.sh https://github.com/conda-forge/miniforge/releases/download/24.9.2-0/Mambaforge-24.9.2-0-Linux-x86_64.sh > conda_install_log.txt 2>&1
Bitte überprüfe diese URL in deinem Browser, um sicherzustellen, dass sie noch gültig ist und die Datei heruntergeladen werden kann.


In [5]:
import os

if os.path.exists('amber_install_log.txt'):
    with open('amber_install_log.txt', 'r') as f:
        print(f.read())
else:
    print("Error: amber_install_log.txt not found. Please ensure cell LsJNdVE87Go2 was executed successfully.")

warning  libmamba Cache file "/usr/local/pkgs/cache/497deca9.json" was modified by another program
warning  libmamba Cache file "/usr/local/pkgs/cache/09cdf8bf.json" was modified by another program
warning  libmamba To upgrade python we need to reinstall noarch
warning  libmamba To upgrade python we need to reinstall noarch
No package record found for ca-certificates-2024.8.30-hbcca054_0.conda!
No package record found for python_abi-3.12-5_cp312.conda!
No package record found for libuuid-2.38.1-h0b41bf4_0.conda!
No package record found for python-3.12.7-hc5c86c4_0_cpython.conda!
No package record found for jsonpointer-3.0.0-py312h7900ff3_1.conda!
No package record found for certifi-2024.8.30-pyhd8ed1ab_0.conda!
No package record found for ruamel.yaml.clib-0.2.8-py312h66e93f0_1.conda!
No package record found for menuinst-2.2.0-py312h7900ff3_0.conda!
No package record found for frozendict-2.4.6-py312h66e93f0_0.conda!
No package record found for brotli-python-1.1.0-py312h2ec8cdc_2.conda!


In [6]:
import os

if os.path.exists('igfold_install_log.txt'):
    with open('igfold_install_log.txt', 'r') as f:
        print(f.read())
else:
    print("Error: igfold_install_log.txt not found. Please ensure cell LsJNdVE87Go2 was executed successfully.")

Found existing installation: torch 2.9.1
Uninstalling torch-2.9.1:
  Successfully uninstalled torch-2.9.1
Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
Looking in indexes: https://download.pytorch.org/whl/cu121
ERROR: Could not find a version that satisfies the requirement torch>=2.6.0 (from versions: 2.2.0+cu121, 2.2.1+cu121, 2.2.2+cu121, 2.3.0+cu121, 2.3.1+cu121, 2.4.0+cu121, 2.4.1+cu121, 2.5.0+cu121, 2.5.1+cu121)
ERROR: No matching distribution found for torch>=2.6.0
Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
  Obtaining dependency information for torch>=1.7.1 from https://files.pythonhosted.org/packages/19/17/e377a460603132b00760511299fceba4102bd95db1a0ee788da21298ccff/torch-2.9.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata
  Using cached torch-2.9.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (30 kB)
Using cached torch-2.9.1-cp312-cp312-manylinux_2_28_x86_64.whl (899.7 MB)
  changing mode of /usr/local/bi

In [9]:
#@title Predict antibody structure with IgFold

if f"/usr/local/lib/python{python_version}/site-packages/" not in sys.path:
    sys.path.insert(0, f"/usr/local/lib/python{python_version}/site-packages/")

import torch
# Import the necessary classes and add them to PyTorch's safe globals
from transformers.models.bert import configuration_bert, tokenization_bert
from transformers import tokenization_utils
torch.serialization.add_safe_globals([configuration_bert.BertConfig, tokenization_bert.BertTokenizer, tokenization_utils.Trie, tokenization_bert.BasicTokenizer, tokenization_bert.WordpieceTokenizer])

from igfold.utils.visualize import *
from igfold import IgFoldRunner

num_models = 1 if single_model else 4
igfold = IgFoldRunner(num_models=num_models)

pred_pdb = os.path.join(pred_dir, f"{name}.pdb")
pred = igfold.fold(
    pred_pdb,
    sequences=sequences,
    do_refine=do_refine,
    use_openmm=True,
    do_renum=do_renum,
)
show_pdb(pred_pdb, len(sequences), bb_sticks=False, sc_sticks=True, color="rainbow")


    The code, data, and weights for this work are made available for non-commercial use 
    (including at commercial entities) under the terms of the JHU Academic Software License 
    Agreement. For commercial inquiries, please contact awichma2[at]jhu.edu.
    License: https://github.com/Graylab/IgFold/blob/main/LICENSE.md
    
Loading 4 IgFold models...
Using device: cpu
Loading /usr/local/lib/python3.12/dist-packages/igfold/trained_models/IgFold/igfold_1.ckpt...
Loading /usr/local/lib/python3.12/dist-packages/igfold/trained_models/IgFold/igfold_2.ckpt...
Loading /usr/local/lib/python3.12/dist-packages/igfold/trained_models/IgFold/igfold_3.ckpt...
Loading /usr/local/lib/python3.12/dist-packages/igfold/trained_models/IgFold/igfold_5.ckpt...
Successfully loaded 4 IgFold models.


BertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


Loaded AntiBERTy model.


/usr/local/lib/python3.12/dist-packages/igfold/utils/coordinates.py:18: UserWarning: Using torch.cross without specifying the dim arg is deprecated.
Please either pass the dim explicitly or simply use torch.linalg.cross.
The default value of dim will change to agree with that of linalg.cross in a future release. (Triggered internally at /pytorch/aten/src/ATen/native/Cross.cpp:63.)
  n_vec = (b_coord - a_coord).expand(bc_vec.shape).cross(bc_vec)
/usr/local/lib/python3.12/dist-packages/igfold/model/components/IPABlock.py:160: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with disable_tf32(), autocast(enabled = False):
/usr/local/lib/python3.10/site-packages/openmm/__init__.py:16: SyntaxWarning: invalid escape sequence '\p'
  os.environ['PATH'] = '%(lib)s;%(lib)s\plugins;%(path)s' % {
/usr/local/lib/python3.10/site-packages/openmm/openmm.py:6921: SyntaxWarning: invalid escape sequence '\S'
  match = re.search("

NameError: name 'exit' is not defined

In [8]:
try:
    import igfold
    print("igfold wurde erfolgreich installiert.")
except ModuleNotFoundError:
    print("Fehler: igfold wurde nicht gefunden. Bitte stelle sicher, dass die Installation erfolgreich war.")

igfold wurde erfolgreich installiert.


/usr/local/lib/python3.12/dist-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


In [32]:
#@title Plot per-residue predicted RMSD

prmsd_fig_file = os.path.join(pred_dir, f"{name}_prmsd.png")
plot_prmsd(sequences, pred.prmsd.cpu(), prmsd_fig_file, shade_cdr=do_renum, pdb_file=pred_pdb)

NameError: name 'pred' is not defined

In [ ]:
#@title Show predicted structure with predicted RMSD

#@markdown Structure is colored from low (blue) to high (red) pRMSD.

show_pdb(pred_pdb, len(sequences), bb_sticks=False, sc_sticks=True, color="b")

In [ ]:
#@title Download results

#@markdown Download zip file containing structure prediction and annotation results. If download fails, results are also accessible from file explorer on the left panel of the notebook.

from google.colab import files
import locale
locale.getpreferredencoding = lambda: "UTF-8"

!zip -FSr $name".result.zip" $pred_dir/ &> /dev/null
files.download(f"{name}.result.zip")

## Summary:

### Data Analysis Key Findings
- The installation command for `igfold` was modified to include the `--verbose` flag and subsequently executed. This action was taken to obtain detailed output during the installation process.

### Insights or Next Steps
- The verbose output from the `igfold` installation attempt will be crucial in diagnosing and identifying the root cause of any previous installation failures.


# Task
Successfully predict and analyze an antibody structure using IgFold, including full installation of all necessary dependencies (Mamba, IgFold, OpenMM, AbNumber) in a Python 3.10 environment, and generate visualizations of the predicted structure and its per-residue RMSD.

## Adjust Install Dependencies Cell (LsJNdVE87Go2) - Final Version

### Subtask:
Modify the 'Install dependencies' cell (LsJNdVE87Go2) to ensure all installations correctly target the Python 3.10 Mamba environment, specifically addressing the torch version compatibility and dependency resolution issues.


**Reasoning**:
The subtask requires modifying the 'Install dependencies' cell (LsJNdVE87Go2) to ensure all installations correctly target the Python 3.10 Mamba environment by adjusting the pip commands and removing the specific index-url for torch. I will replace the existing code in that cell with the updated version.



In [3]:
#@title Install dependencies hallo

# Use system python version (currently 3.12 in Colab)
# The python_version variable is defined in the first input cell

if not os.path.isfile("CONDA_READY"):
  print("installing conda...")
  conda_script_name = "Mambaforge-24.9.2-0-Linux-x86_64.sh"
  # Ensure previous download attempts are cleaned up
  os.system(f"rm -f {conda_script_name}")
  # Try to download and log any wget errors - explicitly output to the desired filename
  wget_cmd = f"wget -O {conda_script_name} https://github.com/conda-forge/miniforge/releases/download/24.9.2-0/{conda_script_name} > conda_install_log.txt 2>&1"
  os.system(wget_cmd)

  # Check if the download was successful
  if not os.path.isfile(conda_script_name):
      # If the file wasn't downloaded, log an explicit error
      with open("conda_install_log.txt", "a") as f:
          f.write(f"ERROR: Failed to download {conda_script_name}. Please check the URL and your internet connection.\n")
      print(f"ERROR: Mambaforge installer not downloaded. See conda_install_log.txt for details.")
  else:
      # If downloaded, proceed with installation
      os.system(f"bash {conda_script_name} -bfp /usr/local >> conda_install_log.txt 2>&1")
      # Add mamba to PATH before trying to use it
      os.environ["PATH"] = "/usr/local/bin:" + os.environ["PATH"]
      os.system("mamba config --set auto_update_conda false >> conda_install_log.txt 2>&1") # Log this too
      os.system("touch CONDA_READY")
      print("Conda installation process completed. See conda_install_log.txt for full details.")
  os.system("cat conda_install_log.txt") # Always print the log at the end for immediate feedback.

if not os.path.isfile("CODE_READY"):
  print("installing igfold...")
  # Force uninstall existing torch/torchvision to ensure clean slate for new version
  os.system(f"python -m pip uninstall -y torch torchvision > igfold_install_log.txt 2>&1")

  # Install torch and torchvision (from specified index) first, explicitly requiring >= 2.6
  # Use /usr/bin/python3 to target Colab's default Python 3.12, and use its globally defined python_version for consistency
  os.system(f"python -m pip install --verbose --force-reinstall 'torch>=2.6.0' torchvision >> igfold_install_log.txt 2>&1")

  # Then install igfold (from PyPI)
  os.system(f"python -m pip install --verbose 'igfold>=0.3.0' >> igfold_install_log.txt 2>&1")

  # Install other dependencies, appending their output to the log
  os.system(f"python -m pip install -q --no-warn-conflicts 'py3Dmol>=2.0.1' matplotlib seaborn >> igfold_install_log.txt 2>&1")

  # Finally, print the combined log and create the ready file
  os.system("cat igfold_install_log.txt")
  os.system("touch CODE_READY")

if do_refine and not os.path.isfile("AMBER_READY"):
  print("installing amber...")
  # Removed openmm version constraint to allow mamba to find compatible version with active Python
  os.system(f"mamba install -y -q -c conda-forge openmm python='{python_version}' pdbfixer > amber_install_log.txt 2>&1") # Use active python_version
  os.system("cat amber_install_log.txt") # Print the log file
  os.system("touch AMBER_READY")

if do_renum and not os.path.isfile("ABNUMBER_READY"):
  print("installing abnumber...")
  # Added -y flag for non-interactive installation and redirect output to abnumber_install_log.txt
  os.system(f"mamba install -y -q -c bioconda abnumber python='{python_version}' > abnumber_install_log.txt 2>&1") # Use active python_version
  os.system(f"python -m pip install pandas --force-reinstall >> abnumber_install_log.txt 2>&1") # Also capture pandas install
  os.system("cat abnumber_install_log.txt") # Print the log file
  os.system("touch ABNUMBER_READY")

installing conda...
Conda installation process completed. See conda_install_log.txt for full details.
installing igfold...
installing amber...
installing abnumber...


## Manual: Clean Restart and Re-Install

### Subtask:
Perform a clean restart of the environment and re-install all dependencies using the updated configuration.


## Manual: Clean Restart and Re-Install

### Subtask:
Perform a clean restart of the environment and re-install all dependencies using the updated configuration.

#### Instructions
1. Execute cell `a126398d` to remove all '.READY' files, ensuring a fresh start for dependency installations.
2. Restart the Colab runtime by navigating to `Runtime` in the menu bar and selecting `Restart runtime`. This step is crucial for the new Python environment and PATH changes to take effect.
3. Execute the modified 'Install dependencies' cell (`LsJNdVE87Go2`) to perform the full re-installation of all packages with the corrected settings.

## Manual: Clean Restart and Re-Install

### Subtask:
Perform a clean restart of the environment and re-install all dependencies using the updated configuration.

#### Instructions
1. Execute cell `a126398d` to remove all '.READY' files, ensuring a fresh start for dependency installations.
2. Restart the Colab runtime by navigating to `Runtime` in the menu bar and selecting `Restart runtime`. This step is crucial for the new Python environment and PATH changes to take effect.
3. Execute the modified 'Install dependencies' cell (`LsJNdVE87Go2`) to perform the full re-installation of all packages with the corrected settings.

## Manual: Verify IgFold Installation

### Subtask:
Verify that IgFold has been successfully installed by attempting to import it.


**Reasoning**:
To verify the successful installation of IgFold, I will execute cell `c34331a5`, which attempts to import the `igfold` module and reports the outcome.



# Task
Befolgen Sie die Anweisungen, um die Installation der `amber`-Abhängigkeit von `OpenMM` und `pdbfixer` zu beheben, die aufgrund eines `ImportError` fehlgeschlagen ist. Analysieren Sie dazu das Installationsprotokoll, indem Sie den Inhalt von `amber_install_log.txt` überprüfen.

## Check Amber Installation Log

### Subtask:
Execute cell `2d2d5d1c` to display the content of `amber_install_log.txt`. This will help diagnose the `ImportError` related to `OpenMM` and `pdbfixer`.


**Reasoning**:
To diagnose the `ImportError` related to `OpenMM` and `pdbfixer`, I will execute cell `2d2d5d1c` which displays the content of `amber_install_log.txt`.



In [10]:
import os

if os.path.exists('amber_install_log.txt'):
    with open('amber_install_log.txt', 'r') as f:
        print(f.read())
else:
    print("Error: amber_install_log.txt not found. Please ensure cell LsJNdVE87Go2 was executed successfully.")

warning  libmamba Cache file "/usr/local/pkgs/cache/497deca9.json" was modified by another program
warning  libmamba Cache file "/usr/local/pkgs/cache/09cdf8bf.json" was modified by another program
warning  libmamba To upgrade python we need to reinstall noarch
warning  libmamba To upgrade python we need to reinstall noarch
No package record found for ca-certificates-2024.8.30-hbcca054_0.conda!
No package record found for python_abi-3.12-5_cp312.conda!
No package record found for libuuid-2.38.1-h0b41bf4_0.conda!
No package record found for python-3.12.7-hc5c86c4_0_cpython.conda!
No package record found for jsonpointer-3.0.0-py312h7900ff3_1.conda!
No package record found for certifi-2024.8.30-pyhd8ed1ab_0.conda!
No package record found for ruamel.yaml.clib-0.2.8-py312h66e93f0_1.conda!
No package record found for menuinst-2.2.0-py312h7900ff3_0.conda!
No package record found for frozendict-2.4.6-py312h66e93f0_0.conda!
No package record found for brotli-python-1.1.0-py312h2ec8cdc_2.conda!


**Reasoning**:
The previous `mamba install` commands failed to find packages and showed warnings about Python 3.12 when Python 3.10 was specified, indicating an issue with environment activation. To fix this, I will modify the 'Install dependencies' cell (LsJNdVE87Go2) to explicitly source the Mambaforge initialization script and activate the base environment before executing `mamba install` commands, ensuring they run within the correct Python 3.10 environment.



In [11]:
import os
import sys

python_version = "3.10"

name = "my_antibody" #@param {type:"string"}
pred_dir = name
os.makedirs(pred_dir, exist_ok=True)

#@markdown Enter antibody sequences for structure prediction. To predict a nanobody structure (or an individual heavy or light chain), simply provide one sequence.
heavy_sequence = "EVQLVQSGPEVKKPGTSVKVSCKASGFTFMSSAVQWVRQARGQRLEWIGWIVIGSGNTNYAQKFQERVTITRDMSTSTAYMELSSLRSEDTAVYYCAAPYCSSISCNDGFDIWGQGTMVTVS" #@param {type:"string"}
light_sequence = "DVVMTQTPFSLPVSLGDQASISCRSSQSLVHSNGNTYLHWYLQKPGQSPKLLIYKVSNRFSGVPDRFSGSGSGTDFTLKISRVEAEDLGVYFCSQSTHVPYTFGGGTKLEIK" #@param {type:"string"}

sequences = {}
if len(heavy_sequence) > 0:
    sequences["H"] = heavy_sequence
if len(light_sequence) > 0:
    sequences["L"] = light_sequence

#@markdown Perform structural refinement with OpenMM
do_refine = True #@param {type:"boolean"}
#@markdown Renumber predicted antibody structure (Chothia) with AbNumber
do_renum = True #@param {type:"boolean"}
#@markdown Use only a single model for predictions (instead of model ensemble)
single_model = False #@param {type:"boolean"}


# Use system python version (currently 3.12 in Colab)
# The python_version variable is defined in the first input cell

if not os.path.isfile("CONDA_READY"):
  print("installing conda...")
  conda_script_name = "Mambaforge-24.9.2-0-Linux-x86_64.sh"
  # Ensure previous download attempts are cleaned up
  os.system(f"rm -f {conda_script_name}")
  # Try to download and log any wget errors - explicitly output to the desired filename
  wget_cmd = f"wget -O {conda_script_name} https://github.com/conda-forge/miniforge/releases/download/24.9.2-0/{conda_script_name} > conda_install_log.txt 2>&1"
  os.system(wget_cmd)

  # Check if the download was successful
  if not os.path.isfile(conda_script_name):
      # If the file wasn't downloaded, log an explicit error
      with open("conda_install_log.txt", "a") as f:
          f.write(f"ERROR: Failed to download {conda_script_name}. Please check the URL and your internet connection.\n")
      print(f"ERROR: Mambaforge installer not downloaded. See conda_install_log.txt for details.")
  else:
      # If downloaded, proceed with installation
      os.system(f"bash {conda_script_name} -bfp /usr/local >> conda_install_log.txt 2>&1")
      # Add mamba to PATH before trying to use it
      os.environ["PATH"] = "/usr/local/bin:" + os.environ["PATH"]
      os.system("mamba config --set auto_update_conda false >> conda_install_log.txt 2>&1") # Log this too
      os.system("touch CONDA_READY")
      print("Conda installation process completed. See conda_install_log.txt for full details.")
  os.system("cat conda_install_log.txt") # Always print the log at the end for immediate feedback.

if not os.path.isfile("CODE_READY"):
  print("installing igfold...")
  # Force uninstall existing torch/torchvision to ensure clean slate for new version
  os.system(f"python -m pip uninstall -y torch torchvision > igfold_install_log.txt 2>&1")

  # Install torch and torchvision (from specified index) first, explicitly requiring >= 2.6
  # Use /usr/bin/python3 to target Colab's default Python 3.12, and use its globally defined python_version for consistency
  os.system(f"python -m pip install --verbose --force-reinstall 'torch>=2.6.0' torchvision >> igfold_install_log.txt 2>&1")

  # Then install igfold (from PyPI)
  os.system(f"python -m pip install --verbose 'igfold>=0.3.0' >> igfold_install_log.txt 2>&1")

  # Install other dependencies, appending their output to the log
  os.system(f"python -m pip install -q --no-warn-conflicts 'py3Dmol>=2.0.1' matplotlib seaborn >> igfold_install_log.txt 2>&1")

  # Finally, print the combined log and create the ready file
  os.system("cat igfold_install_log.txt")
  os.system("touch CODE_READY")

# Commands for mamba install need to source conda.sh and activate base environment correctly
# This ensures that mamba commands use the installed Mambaforge environment.
conda_env_init_cmd = "source /usr/local/etc/profile.d/conda.sh && conda activate base && "

if do_refine and not os.path.isfile("AMBER_READY"):
  print("installing amber...")
  # Removed openmm version constraint to allow mamba to find compatible version with active Python
  os.system(conda_env_init_cmd + f"mamba install -y -q -c conda-forge openmm python='{python_version}' pdbfixer > amber_install_log.txt 2>&1") # Use active python_version
  os.system("cat amber_install_log.txt") # Print the log file
  os.system("touch AMBER_READY")

if do_renum and not os.path.isfile("ABNUMBER_READY"):
  print("installing abnumber...")
  # Added -y flag for non-interactive installation and redirect output to abnumber_install_log.txt
  os.system(conda_env_init_cmd + f"mamba install -y -q -c bioconda abnumber python='{python_version}' > abnumber_install_log.txt 2>&1") # Use active python_version
  os.system(f"python -m pip install pandas --force-reinstall >> abnumber_install_log.txt 2>&1") # Also capture pandas install
  os.system("cat abnumber_install_log.txt") # Print the log file
  os.system("touch ABNUMBER_READY")

**Reasoning**:
To fix the `mamba install` failures and ensure installations target the correct Python 3.10 Mamba environment, I will update the 'Install dependencies' cell (LsJNdVE87Go2) by adding the `conda_env_init_cmd` prefix to all `mamba install` commands and adjust `pip` commands to use the environment's `python`.



In [12]:
#@title Install dependencies hallo

# Use system python version (currently 3.12 in Colab)
# The python_version variable is defined in the first input cell

if not os.path.isfile("CONDA_READY"):
  print("installing conda...")
  conda_script_name = "Mambaforge-24.9.2-0-Linux-x86_64.sh"
  # Ensure previous download attempts are cleaned up
  os.system(f"rm -f {conda_script_name}")
  # Try to download and log any wget errors - explicitly output to the desired filename
  wget_cmd = f"wget -O {conda_script_name} https://github.com/conda-forge/miniforge/releases/download/24.9.2-0/{conda_script_name} > conda_install_log.txt 2>&1"
  os.system(wget_cmd)

  # Check if the download was successful
  if not os.path.isfile(conda_script_name):
      # If the file wasn't downloaded, log an explicit error
      with open("conda_install_log.txt", "a") as f:
          f.write(f"ERROR: Failed to download {conda_script_name}. Please check the URL and your internet connection.\n")
      print(f"ERROR: Mambaforge installer not downloaded. See conda_install_log.txt for details.")
  else:
      # If downloaded, proceed with installation
      os.system(f"bash {conda_script_name} -bfp /usr/local >> conda_install_log.txt 2>&1")
      # Add mamba to PATH before trying to use it
      os.environ["PATH"] = "/usr/local/bin:" + os.environ["PATH"]
      os.system("mamba config --set auto_update_conda false >> conda_install_log.txt 2>&1") # Log this too
      os.system("touch CONDA_READY")
      print("Conda installation process completed. See conda_install_log.txt for full details.")
  os.system("cat conda_install_log.txt") # Always print the log at the end for immediate feedback.

if not os.path.isfile("CODE_READY"):
  print("installing igfold...")
  # Force uninstall existing torch/torchvision to ensure clean slate for new version
  os.system(f"python -m pip uninstall -y torch torchvision > igfold_install_log.txt 2>&1")

  # Install torch and torchvision (from specified index) first, explicitly requiring >= 2.6
  # Use system python, not explicitly /usr/bin/python3
  os.system(f"python -m pip install --verbose --force-reinstall 'torch>=2.6.0' torchvision >> igfold_install_log.txt 2>&1")

  # Then install igfold (from PyPI)
  os.system(f"python -m pip install --verbose 'igfold>=0.3.0' >> igfold_install_log.txt 2>&1")

  # Install other dependencies, appending their output to the log
  os.system(f"python -m pip install -q --no-warn-conflicts 'py3Dmol>=2.0.1' matplotlib seaborn >> igfold_install_log.txt 2>&1")

  # Finally, print the combined log and create the ready file
  os.system("cat igfold_install_log.txt")
  os.system("touch CODE_READY")

# Commands for mamba install need to source conda.sh and activate base environment correctly
# This ensures that mamba commands use the installed Mambaforge environment.
conda_env_init_cmd = "source /usr/local/etc/profile.d/conda.sh && conda activate base && "

if do_refine and not os.path.isfile("AMBER_READY"):
  print("installing amber...")
  # Removed openmm version constraint to allow mamba to find compatible version with active Python
  os.system(conda_env_init_cmd + f"mamba install -y -q -c conda-forge openmm python='{python_version}' pdbfixer > amber_install_log.txt 2>&1") # Use active python_version
  os.system("cat amber_install_log.txt") # Print the log file
  os.system("touch AMBER_READY")

if do_renum and not os.path.isfile("ABNUMBER_READY"):
  print("installing abnumber...")
  # Added -y flag for non-interactive installation and redirect output to abnumber_install_log.txt
  os.system(conda_env_init_cmd + f"mamba install -y -q -c bioconda abnumber python='{python_version}' > abnumber_install_log.txt 2>&1") # Use active python_version
  os.system(f"python -m pip install pandas --force-reinstall >> abnumber_install_log.txt 2>&1") # Also capture pandas install
  os.system("cat abnumber_install_log.txt") # Print the log file
  os.system("touch ABNUMBER_READY")

**Reasoning**:
The 'Install dependencies' cell (LsJNdVE87Go2) will be updated with the provided code, which includes the `conda_env_init_cmd` for mamba installations and adjusted `pip` command paths, to ensure installations target the correct Python 3.10 Mamba environment.



In [13]:
#@title Install dependencies hallo

# Use system python version (currently 3.12 in Colab)
# The python_version variable is defined in the first input cell

if not os.path.isfile("CONDA_READY"):
  print("installing conda...")
  conda_script_name = "Mambaforge-24.9.2-0-Linux-x86_64.sh"
  # Ensure previous download attempts are cleaned up
  os.system(f"rm -f {conda_script_name}")
  # Try to download and log any wget errors - explicitly output to the desired filename
  wget_cmd = f"wget -O {conda_script_name} https://github.com/conda-forge/miniforge/releases/download/24.9.2-0/{conda_script_name} > conda_install_log.txt 2>&1"
  os.system(wget_cmd)

  # Check if the download was successful
  if not os.path.isfile(conda_script_name):
      # If the file wasn't downloaded, log an explicit error
      with open("conda_install_log.txt", "a") as f:
          f.write(f"ERROR: Failed to download {conda_script_name}. Please check the URL and your internet connection.\n")
      print(f"ERROR: Mambaforge installer not downloaded. See conda_install_log.txt for details.")
  else:
      # If downloaded, proceed with installation
      os.system(f"bash {conda_script_name} -bfp /usr/local >> conda_install_log.txt 2>&1")
      # Add mamba to PATH before trying to use it
      os.environ["PATH"] = "/usr/local/bin:" + os.environ["PATH"]
      os.system("mamba config --set auto_update_conda false >> conda_install_log.txt 2>&1") # Log this too
      os.system("touch CONDA_READY")
      print("Conda installation process completed. See conda_install_log.txt for full details.")
  os.system("cat conda_install_log.txt") # Always print the log at the end for immediate feedback.

if not os.path.isfile("CODE_READY"):
  print("installing igfold...")
  # Force uninstall existing torch/torchvision to ensure clean slate for new version
  os.system(f"python -m pip uninstall -y torch torchvision > igfold_install_log.txt 2>&1")

  # Install torch and torchvision (from specified index) first, explicitly requiring >= 2.6
  # Use system python, not explicitly /usr/bin/python3
  os.system(f"python -m pip install --verbose --force-reinstall 'torch>=2.6.0' torchvision >> igfold_install_log.txt 2>&1")

  # Then install igfold (from PyPI)
  os.system(f"python -m pip install --verbose 'igfold>=0.3.0' >> igfold_install_log.txt 2>&1")

  # Install other dependencies, appending their output to the log
  os.system(f"python -m pip install -q --no-warn-conflicts 'py3Dmol>=2.0.1' matplotlib seaborn >> igfold_install_log.txt 2>&1")

  # Finally, print the combined log and create the ready file
  os.system("cat igfold_install_log.txt")
  os.system("touch CODE_READY")

# Commands for mamba install need to source conda.sh and activate base environment correctly
# This ensures that mamba commands use the installed Mambaforge environment.
conda_env_init_cmd = "source /usr/local/etc/profile.d/conda.sh && conda activate base && "

if do_refine and not os.path.isfile("AMBER_READY"):
  print("installing amber...")
  # Removed openmm version constraint to allow mamba to find compatible version with active Python
  os.system(conda_env_init_cmd + f"mamba install -y -q -c conda-forge openmm python='{python_version}' pdbfixer > amber_install_log.txt 2>&1") # Use active python_version
  os.system("cat amber_install_log.txt") # Print the log file
  os.system("touch AMBER_READY")

if do_renum and not os.path.isfile("ABNUMBER_READY"):
  print("installing abnumber...")
  # Added -y flag for non-interactive installation and redirect output to abnumber_install_log.txt
  os.system(conda_env_init_cmd + f"mamba install -y -q -c bioconda abnumber python='{python_version}' > abnumber_install_log.txt 2>&1") # Use active python_version
  os.system(f"python -m pip install pandas --force-reinstall >> abnumber_install_log.txt 2>&1") # Also capture pandas install
  os.system("cat abnumber_install_log.txt") # Print the log file
  os.system("touch ABNUMBER_READY")

**Reasoning**:
The previous `mamba install` commands failed to find packages and showed warnings about Python 3.12 when Python 3.10 was specified, indicating an issue with environment activation. To fix this, I will update the 'Install dependencies' cell (LsJNdVE87Go2, which is ebc0e5ac) to explicitly source the Mambaforge initialization script and activate the base environment before executing `mamba install` commands, ensuring they run within the correct Python 3.10 environment. This update also adjusts `pip` commands to use the environment's `python`.



# Task
I will update cell `LsJNdVE87Go2` with the revised installation script, which incorporates all the necessary corrections for a robust dependency setup within the Mamba Python 3.10 environment.

```python
edit_code(
    cell_id="LsJNdVE87Go2",
    content="""#@title Install dependencies

# Use system python version (currently 3.12 in Colab)
# The python_version variable is defined in the first input cell

if not os.path.isfile("CONDA_READY"):
  print("installing conda...")
  conda_script_name = "Mambaforge-24.9.2-0-Linux-x86_64.sh"
  # Ensure previous download attempts are cleaned up
  os.system(f"rm -f {conda_script_name}")
  # Try to download and log any wget errors - explicitly output to the desired filename
  wget_cmd = f"wget -O {conda_script_name} https://github.com/conda-forge/miniforge/releases/download/24.9.2-0/{conda_script_name} > conda_install_log.txt 2>&1"
  os.system(wget_cmd)

  # Check if the download was successful
  if not os.path.isfile(conda_script_name):
      # If the file wasn't downloaded, log an explicit error
      with open("conda_install_log.txt", "a") as f:
          f.write(f"ERROR: Failed to download {conda_script_name}. Please check the URL and your internet connection.\\n")
      print(f"ERROR: Mambaforge installer not downloaded. See conda_install_log.txt for details.")
  else:
      # If downloaded, proceed with installation
      os.system(f"bash {conda_script_name} -bfp /usr/local >> conda_install_log.txt 2>&1")
      # Add mamba to PATH before trying to use it
      os.environ["PATH"] = "/usr/local/bin:" + os.environ["PATH"]
      os.system("mamba config --set auto_update_conda false >> conda_install_log.txt 2>&1") # Log this too
      os.system("touch CONDA_READY")
      print("Conda installation process completed. See conda_install_log.txt for full details.")
  os.system("cat conda_install_log.txt") # Always print the log at the end for immediate feedback.

# Commands for mamba install need to source conda.sh and activate base environment correctly
# This ensures that mamba commands use the installed Mambaforge environment.
# Define a robust way to initialize the conda environment and run commands within it
conda_init_script = """
eval "$('/usr/local/bin/conda' 'shell.bash' 'hook')"
conda activate base
"""
# Helper function to run commands within the conda environment
def run_conda_command(command, log_file):
    full_command = f"{conda_init_script} {command} >> {log_file} 2>&1"
    os.system(full_command)

if not os.path.isfile("CODE_READY"):
  print("installing igfold...")
  # Force uninstall existing torch/torchvision to ensure clean slate for new version
  # Use mamba's python
  run_conda_command(f"/usr/local/bin/python -m pip uninstall -y torch torchvision", "igfold_install_log.txt")

  # Install torch and torchvision (explicitly requiring >= 2.6)
  # Use mamba's python
  run_conda_command(f"/usr/local/bin/python -m pip install --verbose --force-reinstall 'torch>=2.6.0' torchvision", "igfold_install_log.txt")

  # Then install igfold (from PyPI)
  # Use mamba's python
  run_conda_command(f"/usr/local/bin/python -m pip install --verbose 'igfold>=0.3.0'", "igfold_install_log.txt")

  # Install other dependencies, appending their output to the log
  # Use mamba's python
  run_conda_command(f"/usr/local/bin/python -m pip install -q --no-warn-conflicts 'py3Dmol>=2.0.1' matplotlib seaborn", "igfold_install_log.txt")

  # Finally, print the combined log and create the ready file
  os.system("cat igfold_install_log.txt")
  os.system("touch CODE_READY")

if do_refine and not os.path.isfile("AMBER_READY"):
  print("installing amber...")
  # Removed openmm version constraint to allow mamba to find compatible version with active Python
  # Use mamba directly for conda-forge packages
  run_conda_command(f"mamba install -y -q -c conda-forge openmm python='{python_version}' pdbfixer", "amber_install_log.txt")
  os.system("cat amber_install_log.txt") # Print the log file
  os.system("touch AMBER_READY")

if do_renum and not os.path.isfile("ABNUMBER_READY"):
  print("installing abnumber...")
  # Added -y flag for non-interactive installation and redirect output to abnumber_install_log.txt
  # Use mamba directly for bioconda packages
  run_conda_command(f"mamba install -y -q -c bioconda abnumber python='{python_version}'", "abnumber_install_log.txt")
  # Use mamba's python for pandas
  run_conda_command(f"/usr/local/bin/python -m pip install pandas --force-reinstall", "abnumber_install_log.txt")
  os.system("cat abnumber_install_log.txt") # Print the log file
  os.system("touch ABNUMBER_READY")
""")
```

## Adjust Install Dependencies Cell (LsJNdVE87Go2) - Final Version

### Subtask:
Modify the 'Install dependencies' cell (LsJNdVE87Go2) to ensure all installations correctly target the Python 3.10 Mamba environment, specifically addressing the torch version compatibility and dependency resolution issues.


**Reasoning**:
To ensure all installations correctly target the Python 3.10 Mamba environment and address previous dependency resolution issues, I will update the code in cell `LsJNdVE87Go2` with the provided revised installation script.



In [14]:
#@title Install dependencies hallo

# Use system python version (currently 3.12 in Colab)
# The python_version variable is defined in the first input cell

if not os.path.isfile("CONDA_READY"):
  print("installing conda...")
  conda_script_name = "Mambaforge-24.9.2-0-Linux-x86_64.sh"
  # Ensure previous download attempts are cleaned up
  os.system(f"rm -f {conda_script_name}")
  # Try to download and log any wget errors - explicitly output to the desired filename
  wget_cmd = f"wget -O {conda_script_name} https://github.com/conda-forge/miniforge/releases/download/24.9.2-0/{conda_script_name} > conda_install_log.txt 2>&1"
  os.system(wget_cmd)

  # Check if the download was successful
  if not os.path.isfile(conda_script_name):
      # If the file wasn't downloaded, log an explicit error
      with open("conda_install_log.txt", "a") as f:
          f.write(f"ERROR: Failed to download {conda_script_name}. Please check the URL and your internet connection.\n")
      print(f"ERROR: Mambaforge installer not downloaded. See conda_install_log.txt for details.")
  else:
      # If downloaded, proceed with installation
      os.system(f"bash {conda_script_name} -bfp /usr/local >> conda_install_log.txt 2>&1")
      # Add mamba to PATH before trying to use it
      os.environ["PATH"] = "/usr/local/bin:" + os.environ["PATH"]
      os.system("mamba config --set auto_update_conda false >> conda_install_log.txt 2>&1") # Log this too
      os.system("touch CONDA_READY")
      print("Conda installation process completed. See conda_install_log.txt for full details.")
  os.system("cat conda_install_log.txt") # Always print the log at the end for immediate feedback.

# Commands for mamba install need to source conda.sh and activate base environment correctly
# This ensures that mamba commands use the installed Mambaforge environment.
conda_env_init_cmd = "source /usr/local/etc/profile.d/conda.sh && conda activate base && "

if not os.path.isfile("CODE_READY"):
  print("installing igfold...")
  # Force uninstall existing torch/torchvision to ensure clean slate for new version
  os.system(f"python -m pip uninstall -y torch torchvision > igfold_install_log.txt 2>&1")

  # Install torch and torchvision (from specified index) first, explicitly requiring >= 2.6
  # Use system python, not explicitly /usr/bin/python3
  os.system(f"python -m pip install --verbose --force-reinstall 'torch>=2.6.0' torchvision >> igfold_install_log.txt 2>&1")

  # Then install igfold (from PyPI)
  os.system(f"python -m pip install --verbose 'igfold>=0.3.0' >> igfold_install_log.txt 2>&1")

  # Install other dependencies, appending their output to the log
  os.system(f"python -m pip install -q --no-warn-conflicts 'py3Dmol>=2.0.1' matplotlib seaborn >> igfold_install_log.txt 2>&1")

  # Finally, print the combined log and create the ready file
  os.system("cat igfold_install_log.txt")
  os.system("touch CODE_READY")

if do_refine and not os.path.isfile("AMBER_READY"):
  print("installing amber...")
  # Removed openmm version constraint to allow mamba to find compatible version with active Python
  os.system(conda_env_init_cmd + f"mamba install -y -q -c conda-forge openmm python='{python_version}' pdbfixer > amber_install_log.txt 2>&1") # Use active python_version
  os.system("cat amber_install_log.txt") # Print the log file
  os.system("touch AMBER_READY")

if do_renum and not os.path.isfile("ABNUMBER_READY"):
  print("installing abnumber...")
  # Added -y flag for non-interactive installation and redirect output to abnumber_install_log.txt
  os.system(conda_env_init_cmd + f"mamba install -y -q -c bioconda abnumber python='{python_version}' > abnumber_install_log.txt 2>&1") # Use active python_version
  os.system(f"python -m pip install pandas --force-reinstall >> abnumber_install_log.txt 2>&1") # Also capture pandas install
  os.system("cat abnumber_install_log.txt") # Print the log file
  os.system("touch ABNUMBER_READY")

**Reasoning**:
The subtask requires modifying the 'Install dependencies' cell (LsJNdVE87Go2) with a revised script that explicitly sources Mambaforge and activates the base environment, ensuring correct Python version targeting and dependency resolution.



In [15]:
#@title Install dependencies

# Use system python version (currently 3.12 in Colab)
# The python_version variable is defined in the first input cell

if not os.path.isfile("CONDA_READY"):
  print("installing conda...")
  conda_script_name = "Mambaforge-24.9.2-0-Linux-x86_64.sh"
  # Ensure previous download attempts are cleaned up
  os.system(f"rm -f {conda_script_name}")
  # Try to download and log any wget errors - explicitly output to the desired filename
  wget_cmd = f"wget -O {conda_script_name} https://github.com/conda-forge/miniforge/releases/download/24.9.2-0/{conda_script_name} > conda_install_log.txt 2>&1"
  os.system(wget_cmd)

  # Check if the download was successful
  if not os.path.isfile(conda_script_name):
      # If the file wasn't downloaded, log an explicit error
      with open("conda_install_log.txt", "a") as f:
          f.write(f"ERROR: Failed to download {conda_script_name}. Please check the URL and your internet connection.\n")
      print(f"ERROR: Mambaforge installer not downloaded. See conda_install_log.txt for details.")
  else:
      # If downloaded, proceed with installation
      os.system(f"bash {conda_script_name} -bfp /usr/local >> conda_install_log.txt 2>&1")
      # Add mamba to PATH before trying to use it
      os.environ["PATH"] = "/usr/local/bin:" + os.environ["PATH"]
      os.system("mamba config --set auto_update_conda false >> conda_install_log.txt 2>&1") # Log this too
      os.system("touch CONDA_READY")
      print("Conda installation process completed. See conda_install_log.txt for full details.")
  os.system("cat conda_install_log.txt") # Always print the log at the end for immediate feedback.

# Commands for mamba install need to source conda.sh and activate base environment correctly
# This ensures that mamba commands use the installed Mambaforge environment.
# Define a robust way to initialize the conda environment and run commands within it
conda_init_script = """
eval "$(\'/usr/local/bin/conda\' \'shell.bash\' \'hook\')"
conda activate base
"""
# Helper function to run commands within the conda environment
def run_conda_command(command, log_file):
    full_command = f"{conda_init_script} {command} >> {log_file} 2>&1"
    os.system(full_command)

if not os.path.isfile("CODE_READY"):
  print("installing igfold...")
  # Force uninstall existing torch/torchvision to ensure clean slate for new version
  # Use mamba's python
  run_conda_command(f"/usr/local/bin/python -m pip uninstall -y torch torchvision", "igfold_install_log.txt")

  # Install torch and torchvision (explicitly requiring >= 2.6)
  # Use mamba's python
  run_conda_command(f"/usr/local/bin/python -m pip install --verbose --force-reinstall \'torch>=2.6.0\' torchvision", "igfold_install_log.txt")

  # Then install igfold (from PyPI)
  # Use mamba's python
  run_conda_command(f"/usr/local/bin/python -m pip install --verbose \'igfold>=0.3.0\'", "igfold_install_log.txt")

  # Install other dependencies, appending their output to the log
  # Use mamba's python
  run_conda_command(f"/usr/local/bin/python -m pip install -q --no-warn-conflicts \'py3Dmol>=2.0.1\' matplotlib seaborn", "igfold_install_log.txt")

  # Finally, print the combined log and create the ready file
  os.system("cat igfold_install_log.txt")
  os.system("touch CODE_READY")

if do_refine and not os.path.isfile("AMBER_READY"):
  print("installing amber...")
  # Removed openmm version constraint to allow mamba to find compatible version with active Python
  # Use mamba directly for conda-forge packages
  run_conda_command(f"mamba install -y -q -c conda-forge openmm python='{python_version}' pdbfixer", "amber_install_log.txt")
  os.system("cat amber_install_log.txt") # Print the log file
  os.system("touch AMBER_READY")

if do_renum and not os.path.isfile("ABNUMBER_READY"):
  print("installing abnumber...")
  # Added -y flag for non-interactive installation and redirect output to abnumber_install_log.txt
  # Use mamba directly for bioconda packages
  run_conda_command(f"mamba install -y -q -c bioconda abnumber python='{python_version}'", "abnumber_install_log.txt")
  # Use mamba's python for pandas
  run_conda_command(f"/usr/local/bin/python -m pip install pandas --force-reinstall", "abnumber_install_log.txt")
  os.system("cat abnumber_install_log.txt") # Print the log file
  os.system("touch ABNUMBER_READY")

**Reasoning**:
The subtask requires modifying the 'Install dependencies' cell (LsJNdVE87Go2) with a revised script that explicitly sources Mambaforge and activates the base environment, ensuring correct Python version targeting and dependency resolution.

